<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 8. Интеграция LLM (Ollama + HuggingFace) — подробная методичка

Эта методичка продолжает Этапы 1–7. Предполагается, что у вас:
- `db`, `qdrant`, `app` в `healthy`, `/health` возвращает 200;
- JWT работает, `/auth/register`, `/auth/login`, `/auth/me` — 200;
- `Supervisor` + `Validator` + `Critic` + `Editor` реализованы (Этап 5), но `_call_llm` в `Critic` и `Editor` — **временные заглушки** (возвращают фиксированные строки);
- `backend/data/translation_memory.py` (полный `TMManager`) и `backend/retrieval/vector_index.py` (Qdrant) созданы (Этап 7), в БД ≥ 1000 TM, Qdrant-коллекция `tm_vectors` заполнена.

Если чего-то нет — вернитесь к Этапу 7.

**Цель этапа:** реальные вызовы LLM вместо заглушек, fallback между Ollama и HuggingFace, `/health` проверяет 4 сервиса, Ollama + `ollama-init` в `docker-compose.yml`.

**Недель:** 1.

**Что вы получите в конце этапа:**

- `backend/utils/llm_client.py` — **новый** файл: `call_ollama_with_model` с retry 3×5 с.
- `backend/data/huggingface_translator.py` — **новый**: `TranslatorHuggingFace` (Seq2Seq NLLB + Causal LM Qwen/Llama).
- `backend/data/translator_manager.py` — **новый**: `translator_manager` с probe `/api/tags` (TTL 30 с) и `get_translator_for_engine`.
- **Удалены** временные `_call_llm` из `Critic` и `Editor` — используются `Agent._call_llm` (реальный вызов).
- `docker-compose.yml` — `ollama`, `ollama-init`, `redis`.
- `backend/api/routes/health.py` — обновлён: `db`, `redis`, `qdrant`, `ollama`; `_ollama_name_matches` для точного сравнения имени модели.
- `backend/api/main.py` — `close_ollama_clients()` в `lifespan.finally` (был `pass`).
- Тесты: `test_llm_client.py`, `test_clean_translation.py`, `test_translator_manager.py`, `test_health_ollama.py`.

**Критерии готовности:**

- [ ] Ollama доступна из контейнера `app` через `ollama:11434`.
- [ ] `ollama-init` скачал модели из `.env` (`LLM_MODEL`, `CRITIC_MODEL`, `EDITOR_MODEL`).
- [ ] HF-модель загружается один раз (`get_hf_llm` singleton, `unload_hf_llm` в фикстурах).
- [ ] `TRANSLATION_ENGINE=huggingface` работает без Ollama.
- [ ] `auto` использует Ollama при доступности и для `KNOWN_LANGUAGES`, иначе HF.
- [ ] `/health` — 503 при падении любого из 4 сервисов.
- [ ] `_ollama_name_matches("qwen2.5:7b", "qwen2.5:7b-instruct")` → `False`.
- [ ] Заглушки `_call_llm` удалены из `Critic` и `Editor` (`__qualname__ == "Agent._call_llm"`).
- [ ] Ветка `week-8` создана, коммит сделан.

> **Важно.** 3 файла уже созданы в **Этапе 5**: `http_client.py`, `hf_llm.py`, `llm_processing.py`. Сейчас они **включаются в бой** (были импортированы, но `Critic._call_llm` и `Editor._call_llm` — заглушки, которые не давали им выполниться). В Этапе 8 — **удаление заглушек** (8.6). Если файлов нет — вернитесь к 5.2–5.4.

---

## 8.0. Подготовка: ветка и проверка зависимостей

```powershell
cd D:\RUNG
git checkout -b week-8
```

Проверьте `.env` (в `infrastructure/`):

```env
TRANSLATION_ENGINE=auto
FALLBACK_TO_HF_ON_ERROR=true
AGENT_ENGINE=ollama
LLM_MODEL=qwen2.5:14b
CRITIC_MODEL=qwen2.5:14b
EDITOR_MODEL=qwen2.5:7b
ENSEMBLE_MODELS=qwen2.5:7b,mistral:7b-instruct,llama3.1:8b
HF_TRANSLATION_MODEL=facebook/nllb-200-distilled-1.3B
HF_CRITIC_MODEL=
HF_EDITOR_MODEL=
REDIS_URL=redis://redis:6379/0
```

`HF_CRITIC_MODEL=` и `HF_EDITOR_MODEL=` пустые — `Critic`/`Editor` при `engine="huggingface"` не найдут модель, уйдут в fallback на Ollama (warning в логе). Это норма: HF-модели для Critic/Editor не критичны, Ollama-qwen справляется с JSON лучше.

Проверьте `pyproject.toml`:

```toml
transformers = "^5.15.1"
torch = "^2.13.0"
sentencepiece = "^0.2.2"
```

`sentencepiece` — нужен для NLLB-токенизатора. Без него `AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-1.3B")` упадёт с `ImportError: sentencepiece is not installed`.

---

## 8.1. Что уже есть из Этапа 5 (проверка, не создание)

**1. `backend/utils/http_client.py`** — per-loop `httpx.AsyncClient`. Создан в 5.2. Проверьте:

```powershell
python -m poetry run python -c "from backend.utils.http_client import get_ollama_client, close_ollama_clients; print('OK')"
```

**Что вы должны увидеть:** `OK`.

Что он делает (напоминание):
- `get_ollama_client()` — возвращает клиент, привязанный к текущему event loop (`WeakKeyDictionary` + `threading.Lock`).
- `close_ollama_clients()` — закрывает все клиенты при shutdown.

**2. `backend/utils/hf_llm.py`** — `HuggingFaceLLM` + `get_hf_llm` / `unload_hf_llm`. Создан в 5.3. Проверьте:

```powershell
python -m poetry run python -c "from backend.utils.hf_llm import get_hf_llm, unload_hf_llm; print('OK')"
```

**Что вы должны увидеть:** `OK`.

**3. `backend/utils/llm_processing.py`** — `clean_translation`. Создан в 5.4. Проверьте:

```powershell
python -m poetry run python -c "from backend.utils.llm_processing import clean_translation; print(clean_translation('{\"translation\": \"привет\"}'))"
```

**Что вы должны увидеть:** `привет`.

Если любой из трёх файлов отсутствует — вернитесь к 5.2–5.4 и создайте.

---

## 8.2. `backend/utils/llm_client.py` — новый файл

**Что это:** `call_ollama_with_model(prompt, model_name, ...)` — обёртка над Ollama `/api/generate` с retry 3×5 с и `clean_translation` по умолчанию.

**Зачем это нужно.** `Agent._call_llm` (Этап 5) вызывает Ollama **inline** — 1 с retry. `call_ollama_with_model` — для случаев вне агентов: `perform_translation` (Этап 9), `translate_with_model` (ensemble), когда нужен более щедрый retry (5 с) и очистка ответа. Два модуля — разные задачи.

Создайте `backend/utils/llm_client.py`:

```python
# backend/utils/llm_client.py
import asyncio
import logging

import httpx

from backend.core.config import get_settings
from backend.utils.http_client import get_ollama_client
from backend.utils.llm_processing import clean_translation

logger = logging.getLogger(__name__)
settings = get_settings()

_DEFAULT_SEED = 42


async def call_ollama_with_model(
    prompt: str,
    model_name: str,
    temperature: float = 0.3,
    max_retries: int = 3,
    retry_delay: float = 5.0,
    clean: bool = True,
    seed: int = _DEFAULT_SEED,
) -> str:
    if model_name.startswith("ollama/"):
        model_name = model_name.replace("ollama/", "")

    options = {"temperature": temperature}
    if seed is not None:
        options["seed"] = seed

    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": options,
    }

    client = get_ollama_client()

    for attempt in range(max_retries):
        try:
            response = await client.post(settings.ollama_url, json=payload)
            response.raise_for_status()
            data = response.json()
            raw = data.get("response", "")
            if not raw:
                logger.warning("Empty response from Ollama for model %s", model_name)
                return ""
            result = clean_translation(raw) if clean else raw
            logger.debug(
                "Ollama response from %s: '%s...'", model_name, result[:50]
            )
            return result
        except (
            httpx.TimeoutException,
            httpx.HTTPStatusError,
            httpx.ConnectError,
        ) as e:
            if attempt < max_retries - 1:
                logger.warning(
                    "Ollama error (attempt %d/%d) for model %s: %s. "
                    "Retrying in %ss...",
                    attempt + 1,
                    max_retries,
                    model_name,
                    e,
                    retry_delay,
                )
                await asyncio.sleep(retry_delay)
            else:
                logger.error(
                    "All %d attempts failed for model %s", max_retries, model_name
                )
                raise

    raise RuntimeError(
        f"Failed to get response from Ollama for model {model_name}"
    )
```

### Разбор ключевых решений

**Почему `.replace("ollama/", "")` — в начале.** В `.env` можно задать `LLM_MODEL=ollama/qwen2.5:14b` (префикс для LiteLLM-совместимости). Ollama API понимает только `qwen2.5:14b`, без префикса. Снимаем.

**Почему `retry_delay=5.0`, а не 1.0 (как в `Agent._call_llm`).** `Agent._call_llm` — внутри агентов, где Ollama уже держит модель в памяти (`OLLAMA_KEEP_ALIVE=30m`). 1 с достаточно. `call_ollama_with_model` — для основного перевода, где Ollama может грузить модель с нуля (третья модель вытесняет первую при `OLLAMA_MAX_LOADED_MODELS=2`). Загрузка 14B — 20–30 с. 5 с — разумный первый retry; 3 попытки покрывают 15 с.

**Почему `except (TimeoutException, HTTPStatusError, ConnectError)`, а не `Exception`.** Три типа `httpx`-исключений покрывают сетевые сбои. Другие (`ValueError` из `json()`, `KeyError`) — баги, retry их не исправит.

**Почему `if not raw: return ""` без retry.** Пустой ответ — не сетевой сбой, а «модель не сгенерировала». Retry даст тот же пустой результат. `""` — caller обработает.

**Почему `clean=True` по умолчанию.** В 90% случаев нужен чистый перевод. `clean=False` — для отладки промптов (сырой ответ).

**Почему `raise RuntimeError(...)` после цикла.** Недостижимо (цикл либо `return`, либо `raise` в `except`), но защищает от «забыли raise». Без — функция могла бы вернуть `None`.

**Почему `settings` — module-level `get_settings()`.** Один раз при импорте. В тестах — `importlib.reload` после `monkeypatch.setenv`.

**Почему **не** используется в `Agent._call_llm`.** `Agent._call_llm` уже имеет `@retry(max_attempts=3, delay=1.0)` и делает `get_ollama_client().post(...)` напрямую. Замена дала бы двойной retry (3 × 3 = 9). Два модуля — разные задачи.

### Проверка

```powershell
python -m poetry run python -c "import asyncio; from backend.utils.llm_client import call_ollama_with_model; async def m(): r = await call_ollama_with_model('Translate to Russian: Hello', 'qwen2.5:7b'); print('Result:', r[:50]); asyncio.run(m())"
```

**Что вы должны увидеть** (если Ollama ещё не поднята — `ConnectError` через 15 с; поднимем в 8.6):

```
Result: Привет
```

### Коммит

```powershell
git add backend/utils/llm_client.py
git commit -m "feat(week-8): add call_ollama_with_model with 5s retry"
```

---

## 8.3. `backend/data/huggingface_translator.py` — новый файл

**Что это:** `TranslatorHuggingFace` — обёртка над HF-моделями для **перевода** (не для генерации агентами). Два типа: Seq2Seq (NLLB, mBART) и Causal LM (Qwen, Llama).

**Зачем это нужно:** `translator_manager` (8.4) выбирает HF, когда Ollama недоступна или пара языков не в `KNOWN_LANGUAGES` (`udm`, `os`, `av`, `tk`). HF NLLB-200 знает 200 языков, включая все 18 проекта. Causal LM (Qwen) — для instruction-style перевода.

Создайте `backend/data/huggingface_translator.py`:

```python
# backend/data/huggingface_translator.py
import asyncio
import logging
from typing import Dict, Optional

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

from backend.core.config import get_settings

logger = logging.getLogger(__name__)
settings = get_settings()


class TranslatorHuggingFace:
    """
    Async wrapper for Hugging Face translation models.

    Supports Seq2Seq (NLLB, mBART) and Causal LM (Qwen, Llama) checkpoints.
    Device and dtype are picked once at load time: CUDA uses float16, MPS
    and CPU use float32. Loading a 1.3B-parameter model in float32 on CPU
    makes interactive fallback unusable — several minutes per sentence —
    so the device-specific dtype is not optional here.
    """

    NLLB_LANG_CODES: Dict[str, str] = {
        "en": "eng_Latn",
        "ru": "rus_Cyrl",
        "tg": "tgk_Cyrl",
        "tt": "tat_Cyrl",
        "fa": "pes_Arab",
        "ar": "arb_Arab",
        "tr": "tur_Latn",
        "uz": "uzn_Latn",
        "kk": "kaz_Cyrl",
        "ky": "kir_Cyrl",
        "bg": "bul_Cyrl",
        "id": "ind_Latn",
        "ms": "zsm_Latn",
        "os": "oss_Cyrl",
        "av": "ava_Cyrl",
        "ba": "bak_Cyrl",
        "tk": "tuk_Latn",
        "udm": "udm_Cyrl",
    }

    def __init__(self, model_name: Optional[str] = None):
        self.model_name = model_name or settings.HF_TRANSLATION_MODEL
        self.tokenizer = None
        self.model = None
        self.model_type: Optional[str] = None
        self.device = self._pick_device()
        self._lock = asyncio.Lock()

    @staticmethod
    def _pick_device() -> str:
        if torch.cuda.is_available():
            return "cuda"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
        return "cpu"

    def _pick_dtype(self) -> torch.dtype:
        # float16 on CUDA; MPS and CPU stay in float32 for numerical
        # stability and because MPS does not support fp16 reliably for
        # every operator used by these models.
        if self.device == "cuda":
            return torch.float16
        return torch.float32

    def _map_lang_for_seq2seq(self, lang: str) -> str:
        return self.NLLB_LANG_CODES.get(lang, lang)

    async def _load_model(self) -> None:
        async with self._lock:
            if self.model is not None:
                return

            logger.info(
                "Loading Hugging Face model '%s' on device=%s ...",
                self.model_name,
                self.device,
            )
            loop = asyncio.get_running_loop()

            def _load():
                tokenizer = AutoTokenizer.from_pretrained(
                    self.model_name,
                    token=settings.HF_TOKEN,
                    trust_remote_code=False,
                )
                try:
                    model = AutoModelForSeq2SeqLM.from_pretrained(
                        self.model_name,
                        token=settings.HF_TOKEN,
                        trust_remote_code=False,
                        torch_dtype=self._pick_dtype(),
                    )
                    return tokenizer, model, "seq2seq"
                except Exception:
                    logger.info(
                        "Model '%s' is not Seq2Seq, loading as Causal LM",
                        self.model_name,
                    )
                    model = AutoModelForCausalLM.from_pretrained(
                        self.model_name,
                        token=settings.HF_TOKEN,
                        trust_remote_code=False,
                        torch_dtype=self._pick_dtype(),
                    )
                    return tokenizer, model, "causal"

            try:
                self.tokenizer, self.model, self.model_type = (
                    await loop.run_in_executor(None, _load)
                )
            except Exception:
                logger.exception(
                    "Failed to load model '%s'", self.model_name
                )
                raise

            self.model.eval()
            try:
                self.model.to(self.device)
            except Exception as e:
                logger.warning(
                    "Could not move '%s' to %s (%s); staying on CPU",
                    self.model_name,
                    self.device,
                    e,
                )
                self.device = "cpu"

            logger.info(
                "Hugging Face model loaded (type=%s, device=%s)",
                self.model_type,
                self.device,
            )

    async def translate(
        self,
        text: str,
        source_lang: str = "en",
        target_lang: str = "ru",
    ) -> str:
        await self._load_model()
        loop = asyncio.get_running_loop()

        if self.model_type == "seq2seq":
            return await self._translate_seq2seq(
                text, source_lang, target_lang, loop
            )
        return await self._translate_causal(
            text, source_lang, target_lang, loop
        )

    async def _translate_seq2seq(self, text, src_lang, tgt_lang, loop):
        src_code = self._map_lang_for_seq2seq(src_lang)
        tgt_code = self._map_lang_for_seq2seq(tgt_lang)

        try:
            forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(tgt_code)
        except Exception:
            forced_bos_token_id = None

        def _generate() -> str:
            tokenizer_kwargs = {
                "return_tensors": "pt",
                "truncation": True,
                "max_length": 512,
            }
            if hasattr(self.tokenizer, "src_lang"):
                tokenizer_kwargs["src_lang"] = src_code

            inputs = self.tokenizer(text, **tokenizer_kwargs)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            generate_kwargs = {
                "max_length": 512,
                "num_beams": 4,
                "early_stopping": True,
            }
            if forced_bos_token_id is not None:
                generate_kwargs["forced_bos_token_id"] = forced_bos_token_id

            with torch.inference_mode():
                outputs = self.model.generate(**inputs, **generate_kwargs)
            return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        try:
            result = await loop.run_in_executor(None, _generate)
            logger.info(
                "HF Seq2Seq translation (%s->%s): %s...",
                src_lang, tgt_lang, result[:50],
            )
            return result
        except Exception as e:
            logger.error("Hugging Face Seq2Seq translation error: %s", e)
            raise

    async def _translate_causal(self, text, src_lang, tgt_lang, loop):
        prompt = f"Translate from {src_lang} to {tgt_lang}: {text}\nTranslation:"

        def _generate() -> str:
            inputs = self.tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=512
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                )
            full_output = self.tokenizer.decode(
                outputs[0], skip_special_tokens=True
            )
            marker = "Translation:"
            if marker in full_output:
                return full_output.split(marker)[-1].strip()
            return full_output.strip()

        try:
            result = await loop.run_in_executor(None, _generate)
            logger.info(
                "HF Causal translation (%s->%s): %s...",
                src_lang, tgt_lang, result[:50],
            )
            return result
        except Exception as e:
            logger.error("Hugging Face Causal translation error: %s", e)
            raise
```

### Разбор ключевых решений

**Почему `NLLB_LANG_CODES` — 18 языков.** NLLB-200 знает 200, но API требует `eng_Latn`-формат. Маппинг — единственное место для соответствия. Если добавите 19-й язык — нужно добавить и здесь, иначе `convert_tokens_to_ids` вернёт `unk_token_id`.

**Почему fallback `return self.NLLB_LANG_CODES.get(lang, lang)`.** Если языка нет — возвращаем ISO-код. `convert_tokens_to_ids("new")` → `unk_token_id` (2) → `forced_bos_token_id=2` → перевод на языке по умолчанию. Плохо, но не падает.

**Почему **сначала** `AutoModelForSeq2SeqLM`, **потом** `AutoModelForCausalLM`.** Большинство моделей для перевода — Seq2Seq (NLLB, mBART). Causal LM (Qwen) — «на крайний случай», но тоже работает. Seq2Seq-first — без лишних попыток для 95% случаев.

**Почему `trust_remote_code=False`.** NLLB/mBART — стандартные архитектуры, custom code не нужен. `True` разрешил бы **выполнение произвольного Python-кода** при загрузке — RCE. `False` — безопасный дефолт.

**Почему `_pick_dtype`: float16 только для CUDA.** fp16 на CPU не поддерживается частью операторов `torch` (`RuntimeError`). MPS — нестабильно для 1.3B-моделей (OOM, неподдерживаемые операторы). float32 на CPU/MPS — медленнее, но работает.

**Почему `model.to(self.device)` в try/except.** CUDA OOM, MPS не поддерживает оператор, `cuda:0` не существует — всё ловится, `self.device = "cpu"` — fallback. Медленнее, но работает.

**Почему `if hasattr(self.tokenizer, "src_lang")`.** NLLB-токенизатор имеет `src_lang`. T5 — нет. `hasattr` — проверка, что модель поддерживает указание `src_lang`. Без — `tokenizer(text, ...)` без `src_lang`.

**Почему `forced_bos_token_id = convert_tokens_to_ids(tgt_code)`.** NLLB использует special token начала перевода для целевого языка. `convert_tokens_to_ids("rus_Cyrl")` → id → `generate(forced_bos_token_id=id)` — модель генерирует на русском. Если токена нет — `unk_token_id`, плохой перевод, но не падение.

**Почему `num_beams=4` + `early_stopping=True`.** Beam search с 4 лучами — качество лучше greedy (1 луч), быстрее 8. `early_stopping` — останавливает, когда все лучи достигли EOS.

**Почему `max_length=512`.** NLLB-200-distilled-1.3B — контекст 1024. 512 — половина для входа, 512 для выхода. Стандартный сегмент (2–3 предложения) влезает.

**Почему `torch.inference_mode()`, а не `torch.no_grad()`.** `inference_mode` — быстрее на 10–15%, строже (запрещает autograd). Для inference — правильный выбор.

**Почему `_translate_causal` ищет `"Translation:"` в `full_output`.** Causal LM (Qwen) возвращает `prompt + generated` = `"Translate from en to ru: Hello\nTranslation: Привет"`. Нужно отрезать prompt. `marker in full_output` — проверка, что модель не срезала маркер.

**Почему `do_sample=False` для causal.** Deterministic — один вход → один выход. Для перевода — правильно. `True` даёт случайность — невоспроизводимо.

**Почему `_lock = asyncio.Lock()` в `__init__`.** Один lock на экземпляр. Если создать в `_load_model`, каждый вызов создал бы новый lock — race condition. `self._lock` — instance attribute.

**Почему **не** singleton.** `translator_manager._get_hf_translator` создаёт **один** экземпляр lazily. Singleton — только для `HuggingFaceLLM` (для агентов). `TranslatorHuggingFace` — единственный экземпляр в `translator_manager`.

**Почему `logger.exception(...)` в `except` вокруг загрузки.** Автоматически включает стектрейс. Короче, чем `logger.error(..., exc_info=True)`.

### Коммит

```powershell
git add backend/data/huggingface_translator.py
git commit -m "feat(week-8): add TranslatorHuggingFace (NLLB + Causal LM)"
```

---

## 8.4. `backend/data/translator_manager.py` — новый файл

**Что это:** `translator_manager` — singleton, который решает: Ollama или HuggingFace для конкретного запроса. Кэширует доступность Ollama на 30 с.

**Зачем это нужно:** `perform_translation` (Этап 9) вызывает `translator_manager.get_translator_for_engine(engine, src, tgt)`. Возвращает:
- `None` → использовать Ollama (прямой вызов).
- `Callable` → использовать HF (NLLB, Causal).

Создайте `backend/data/translator_manager.py`:

```python
# backend/data/translator_manager.py
import logging
import time
from typing import Any, Callable, Optional

from backend.core.config import get_settings
from backend.core.constants import KNOWN_LANGUAGES
from backend.core.exceptions import EngineUnavailableError
from backend.data.huggingface_translator import TranslatorHuggingFace
from backend.utils.http_client import get_ollama_client

logger = logging.getLogger(__name__)
settings = get_settings()

_OLLAMA_PROBE_TIMEOUT = 2.0
_OLLAMA_PROBE_TTL = 30.0


class TranslatorManager:
    def __init__(self):
        self.hf_translator = None
        self.hf_model = settings.HF_TRANSLATION_MODEL
        self._ollama_available: Optional[bool] = None
        self._ollama_cache_time = 0.0

    def _get_hf_translator(self) -> Callable[..., Any]:
        if self.hf_translator is None:
            logger.info(
                "Initializing Hugging Face translator with model %s",
                self.hf_model,
            )
            self.hf_translator = TranslatorHuggingFace(self.hf_model)
        return self.hf_translator.translate

    async def _is_ollama_available(self) -> bool:
        now = time.monotonic()
        if (
            self._ollama_available is not None
            and now - self._ollama_cache_time < _OLLAMA_PROBE_TTL
        ):
            return self._ollama_available

        tags_url = settings.ollama_url.replace("/api/generate", "/api/tags")
        try:
            client = get_ollama_client()
            resp = await client.get(tags_url, timeout=_OLLAMA_PROBE_TIMEOUT)
            self._ollama_available = resp.status_code == 200
        except Exception:
            self._ollama_available = False

        self._ollama_cache_time = now
        return self._ollama_available

    async def get_translator_for_engine(
        self,
        engine: str,
        source_lang: str,
        target_lang: str,
    ) -> Optional[Callable[..., Any]]:
        ollama_capable = (
            source_lang in KNOWN_LANGUAGES and target_lang in KNOWN_LANGUAGES
        )
        ollama_available = await self._is_ollama_available()
        use_hf = False

        if engine == "ollama":
            if not ollama_capable or not ollama_available:
                if settings.FALLBACK_TO_HF_ON_ERROR:
                    logger.warning(
                        "Ollama unavailable or unsupported, "
                        "falling back to Hugging Face"
                    )
                    use_hf = True
                else:
                    raise EngineUnavailableError(
                        "Ollama requested but unavailable",
                        context={
                            "engine": "ollama",
                            "source_lang": source_lang,
                            "target_lang": target_lang,
                            "ollama_capable": ollama_capable,
                            "ollama_available": ollama_available,
                            "fallback_enabled": settings.FALLBACK_TO_HF_ON_ERROR,
                        },
                    )
        elif engine == "huggingface":
            use_hf = True
        elif engine == "auto":
            if not ollama_capable or not ollama_available:
                use_hf = True
        else:
            logger.warning("Unknown engine '%s', using auto", engine)
            if not ollama_capable or not ollama_available:
                use_hf = True

        if use_hf:
            return self._get_hf_translator()
        return None


translator_manager = TranslatorManager()
```

### Разбор ключевых решений

**Почему `_OLLAMA_PROBE_TIMEOUT = 2.0`, `_OLLAMA_PROBE_TTL = 30.0`.** Probe `/api/tags` — лёгкий (список моделей), < 500 мс. 2 с — запас. TTL 30 с: 100+ запросов без повторного probe; если Ollama упала — до 30 с «фантомной доступности» (потом 5 с probe). Если поднялась — до 30 с «фантомной недоступности». Компромисс.

**Почему `time.monotonic()`, а не `time.time()`.** `monotonic` не прыгает при NTP-синхронизации. `time.time()` может уйти назад → `now - cache_time < 0` → TTL не истечёт. `monotonic` — правильный выбор.

**Почему `/api/tags`, а не `/api/generate`.** `/api/tags` — GET, лёгкий. `/api/generate` — POST, требует JSON, запускает inference (медленно). Probe через `/api/generate` был бы «проверка = загрузка модели = 10 с». `/api/tags` — 100 мс.

**Почему `except Exception` без конкретного типа.** `ConnectError`, `TimeoutException`, `HTTPStatusError`, `RuntimeError` (loop closed) — всё = «Ollama недоступна». Широкий `except` — правильно для «best effort»-проверки.

**Почему `_ollama_available: Optional[bool]`, а не `bool`.** `None` = «не проверяли». `False` = «недоступна». `True` = «доступна». `if self._ollama_available is not None` — проверка «уже проверяли».

**Почему `get_translator_for_engine` возвращает `Optional[Callable]`, а не `str`.** Контракт:
- `None` — Ollama (прямой вызов).
- `Callable` — HF (`translator.translate`).

В Этапе 9: `if translator_func is None: Ollama else: HF`. Просто и явно.

**Почему `engine == "ollama"` с fallback, а не `raise` сразу.** `FALLBACK_TO_HF_ON_ERROR=true` (дефолт в `.env`) — падать бессмысленно, HF работает. `false` — `raise EngineUnavailableError` с контекстом: `{engine, source_lang, target_lang, ollama_capable, ollama_available, fallback_enabled}`. Клиент видит 502 + `code: "engine_unavailable"`.

**Почему `EngineUnavailableError` (наследник `EngineError` → 502), а не `HTTPException`.** 502 Bad Gateway — «upstream (Ollama) недоступен». `RUNGException` → глобальный хендлер → структурированный JSON с `code`/`context`.

**Почему `ollama_capable` = **оба** языка в `KNOWN_LANGUAGES`.** Если **любой** не в списке — HF. `KNOWN_LANGUAGES` (14) — языки, которые qwen2.5 знает надёжно. `udm`/`os`/`av`/`tk` (4 остальных) — только HF.

**Почему `engine == "auto"` **не** бросает при недоступной Ollama.** `auto` = «используй что доступно». Ollama недоступна → HF. Оба недоступны → `translator_manager` вернёт `None` (Ollama), но Ollama недоступна → `call_ollama_with_model` → `ConnectError` → 502. `auto` с `not ollama_available` идёт в HF — если HF-модель не загрузится, `translate` бросит `OSError` → 502 через `_http_error_from_transport` (Этап 9).

**Почему `else` (неизвестный engine) → `auto` с warning.** `Settings.validate_engine` (Этап 2) валидирует `TRANSLATION_ENGINE`. Но `engine` в `perform_translation` — отдельный runtime-параметр. Pydantic `TranslateRequest.engine: Optional[TranslationEngine]` — `Literal["auto", "ollama", "huggingface"]`, отсекает `"google"` на уровне API. Но если кто-то вызывает `perform_translation(engine="google")` напрямую — `else` даст warning + auto-поведение.

**Почему `translator_manager = TranslatorManager()` на module-level.** Singleton. Один `_is_ollama_available` кэш на процесс. Если бы каждый `perform_translation` создавал новый — каждый делал бы probe, кэш бесполезен.

### Проверка (Ollama пока не поднята — будет `unavailable`)

```powershell
python -m poetry run python -c "
import asyncio
from backend.data.translator_manager import translator_manager

async def m():
    # ollama недоступна → HF
    f1 = await translator_manager.get_translator_for_engine('ollama', 'en', 'ru')
    print('ollama+unavailable+fallback:', 'HF' if f1 else 'None')

    # huggingface → всегда HF
    f2 = await translator_manager.get_translator_for_engine('huggingface', 'en', 'ru')
    print('huggingface:', 'HF' if f2 else 'None')

    # auto + udm (не в KNOWN) → HF
    f3 = await translator_manager.get_translator_for_engine('auto', 'en', 'udm')
    print('auto+udm:', 'HF' if f3 else 'None')

asyncio.run(m())
"
```

**Что вы должны увидеть:**

```
ollama+unavailable+fallback: HF
huggingface: HF
auto+udm: HF
```

### Коммит

```powershell
git add backend/data/translator_manager.py
git commit -m "feat(week-8): add TranslatorManager (Ollama/HF routing, 30s probe TTL)"
```

---

## 8.5. Удаление заглушек в `Critic` и `Editor`

**Что это:** в Этапе 5 в `Critic` и `Editor` были добавлены временные `_call_llm`-override, возвращающие фиксированные строки. Сейчас они **удаляются** — используется `Agent._call_llm` (реальный вызов).

**Зачем это нужно:** `Agent._call_llm` (Этап 5.5) реализует:
- **Ollama** — `get_ollama_client().post(settings.ollama_url, ...)`, `@retry(max_attempts=3, delay=1.0)`, `options={temperature, seed=42}`.
- **HF** — `self.hf_llm.generate(prompt, ..., seed=42)`.

Заглушка в `Critic` возвращала `{"overall_score": 5.0, ...}` — фейковая оценка. `Supervisor` видел 5.0 < 7.0 → Editor в цикле. Реальные вызовы — Critic получает настоящий score от LLM.

### Что удалить

**В `backend/agents/critic.py`** — найдите блок:

```python
    # ВРЕМЕННО (убрать в Этапе 8): заглушка, возвращающая валидный JSON
    async def _call_llm(self, prompt: str) -> str:
        return '{"adequacy": 5.0, "fluency": 5.0, "terminology": 5.0, "overall_score": 5.0, "reasoning": "Stub LLM: real call in Этап 8.", "suggestions": [], "missing_terms": []}'
```

**Удалить** все 4 строки (включая комментарий). Метод `_call_llm` унаследуется из `Agent`.

**В `backend/agents/editor.py`** — найдите блок:

```python
    # ВРЕМЕННО (убрать в Этапе 8): заглушка
    async def _call_llm(self, prompt: str) -> str:
        return "Заглушка: перевод не изменён"
```

**Удалить** все 3 строки.

### Проверка

```powershell
python -m poetry run python -c "
from backend.agents.critic import Critic
from backend.agents.editor import Editor
print('Critic._call_llm:', Critic._call_llm.__qualname__)
print('Editor._call_llm:', Editor._call_llm.__qualname__)
"
```

**Что вы должны увидеть** (метод из `Agent`):

```
Critic._call_llm: Agent._call_llm
Editor._call_llm: Agent._call_llm
```

Если видите `Critic._call_llm` — заглушка не удалена.

### Разбор

**Почему простое удаление работает.** Python MRO: `Critic(Agent)`. После удаления override Python ищет `_call_llm` в `Agent` — находит. Никаких других изменений.

**Почему заглушка была в `Critic`/`Editor`, а не в `Agent`.** `Agent._call_llm` — общая реализация. Заглушка — временная, для двух классов. Менять `Agent` ради Этапа 5 = ломать Этап 8. Override в подклассе — правильно.

**Почему `Critic.process` не сломается.** `Critic.process` вызывает `await self._call_llm(prompt)`. Без override — `Agent._call_llm`:
1. `self.engine == "huggingface"` и `self.hf_llm is not None` → `self.hf_llm.generate(...)`.
2. Иначе → `get_ollama_client().post(self.settings.ollama_url, ...)`.

`Critic.__init__` (Этап 5) уже установил `self.hf_llm = None` (Ollama) или `get_hf_llm(...)` (HF). `self.engine` — резолвится в `Agent.__init__`. Всё готово.

**Ловушка:** после удаления unit-тесты `Critic`/`Editor` (Этап 5) упадут, если не мокают `_call_llm`. В тестах — `monkeypatch.setattr(Critic, "_call_llm", AsyncMock(...))`. Без мока — реальный HTTP к Ollama (или `ConnectError`).

### Коммит

```powershell
git add backend/agents/critic.py backend/agents/editor.py
git commit -m "refactor(week-8): remove temporary _call_llm stubs from Critic and Editor"
```

---

## 8.6. `infrastructure/docker-compose.yml` — ollama, ollama-init, redis

**Что это:** добавить три сервиса в `docker-compose.yml` (Этап 3: `db`, `app`; Этап 7: `qdrant`):
- `ollama` — LLM-сервер.
- `ollama-init` — одноразовый, качает модели из `.env`.
- `redis` — нужен для `/health` (полная проверка 4 сервисов) и Celery (Этап 11).

**Зачем `ollama-init`.** `ollama` не скачивает модели автоматически. Без init — вручную `docker compose exec ollama ollama pull qwen2.5:14b`. `ollama-init` ждёт готовности `ollama`, потом `ollama pull` для каждой модели из `.env`. Один раз при первом запуске, потом no-op (модели в volume).

**Зачем redis сейчас.** `/health` (8.7) проверяет redis. Без redis — 503. `redis:7-alpine` — тривиальный контейнер. `worker`/`beat` (Этап 11) подключатся к нему же.

Откройте `infrastructure/docker-compose.yml`. Добавьте:

**1. Сервис `redis` — после `db`:**

```yaml
  redis:
    image: redis:7-alpine
    command: ["redis-server", "--appendonly", "yes"]
    volumes:
      - redis_data:/data
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 10s
      timeout: 5s
      retries: 5
      start_period: 5s
    networks:
      - rung_internal
```

**2. Сервис `ollama` — после `qdrant`:**

```yaml
  ollama:
    image: ollama/ollama:latest
    environment:
      OLLAMA_HOST: 0.0.0.0:11434
      # Keep two models resident: the 14B critic + one worker model.
      # Fits comfortably in the 22G container limit.
      OLLAMA_MAX_LOADED_MODELS: 2
      OLLAMA_NUM_PARALLEL: 2
      # 30 minutes is enough for a burst of requests without reloading
      # the 14B critic on every call.
      OLLAMA_KEEP_ALIVE: 30m
    volumes:
      - ollama_data:/root/.ollama
    ports:
      - "11435:11434"
    restart: unless-stopped
    deploy:
      resources:
        limits:
          memory: 22G
        reservations:
          memory: 12G
    healthcheck:
      test: ["CMD-SHELL", "ollama list >/dev/null 2>&1 || exit 1"]
      interval: 10s
      timeout: 5s
      retries: 30
      start_period: 30s
    networks:
      - rung_internal
```

**3. Сервис `ollama-init` — после `ollama`:**

```yaml
  ollama-init:
    image: ollama/ollama:latest
    environment:
      OLLAMA_HOST: http://ollama:11434
      LLM_MODEL: ${LLM_MODEL}
      CRITIC_MODEL: ${CRITIC_MODEL:-}
      EDITOR_MODEL: ${EDITOR_MODEL:-}
    volumes:
      - ollama_data:/root/.ollama
    depends_on:
      ollama:
        condition: service_healthy
    entrypoint: ["/bin/sh", "-c"]
    # Folded scalar (>-): newlines become spaces, so CRLF line endings
    # in the YAML file cannot break the shell script.
    command: >-
      echo 'Checking Ollama...' &&
      until ollama list >/dev/null 2>&1; do echo 'Ollama is not ready yet...'; sleep 2; done &&
      for MODEL in "$${LLM_MODEL}" "$${CRITIC_MODEL}" "$${EDITOR_MODEL}"; do
        if [ -n "$${MODEL}" ]; then
          echo "Checking model: $${MODEL}";
          if ollama list | grep -q "^$${MODEL}"; then
            echo "Model $${MODEL} already exists";
          else
            echo "Pulling model: $${MODEL}";
            ollama pull "$${MODEL}";
          fi;
        fi;
      done &&
      echo 'Ollama initialization completed'
    restart: "no"
    networks:
      - rung_internal
```

**4. В `app.depends_on` — добавить `redis` и `ollama`:**

```yaml
    depends_on:
      db:
        condition: service_healthy
      redis:
        condition: service_healthy
      qdrant:
        condition: service_healthy
      ollama:
        condition: service_healthy
```

**5. В `volumes:` — добавить `redis_data` и `ollama_data`:**

```yaml
volumes:
  postgres_data:
    name: rung_postgres_data
  redis_data:
    name: rung_redis_data
  qdrant_storage:
    name: rung_qdrant_storage
  ollama_data:
    name: rung_ollama_data
```

### Разбор ключевых решений

**Почему `OLLAMA_MAX_LOADED_MODELS: 2`.** `LLM_MODEL=qwen2.5:14b` + `CRITIC_MODEL=qwen2.5:14b` + `EDITOR_MODEL=qwen2.5:7b` — 3 имени, но `LLM` и `CRITIC` — один и тот же qwen2.5:14b. Реально 2 модели: 14b + 7b. `MAX_LOADED=2` держит обе. `MAX_LOADED=1` — переключение 14b ↔ 7b на каждом Editor — 20–30 с на переключение. `MAX_LOADED=3` — не влезет в 22G (если 3 разные модели).

**Почему `OLLAMA_NUM_PARALLEL: 2`.** Ollama обрабатывает 2 одновременных inference. `_MAX_CONCURRENT_TRANSLATIONS = 4` (Этап 9) — FastAPI отправляет до 4, но Ollama обрабатывает 2, остальные в очереди. `NUM_PARALLEL=4` — 4 × 14b = 80 ГБ, OOM. 2 — баланс.

**Почему `OLLAMA_KEEP_ALIVE: 30m`.** Сколько модель в памяти после последнего запроса. Дефолт 5 мин. Для eval (паузы 2 мин) — модель выгружалась бы, следующий батч грузит 30 с. 30 мин — переживает eval, не держит вечно.

**Почему `ports: "11435:11434"`.** Внешний 11435 → внутренний 11434. Хост-порт 11434 может быть занят локальной Ollama. 11435 — отдельный. Внутри `rung_internal` — `ollama:11434` (внутренний). `11435` — только для отладки.

**Почему `deploy.resources.limits.memory: 22G`, `reservations: 12G`.** `limits` — OOM-kill при превышении. 22G: 14b (~10 ГБ) + 7b (~5 ГБ) + KV-cache (~5 ГБ) + запас. `reservations` — гарантия 12 ГБ для одной 14b-модели.

**Почему healthcheck `ollama list` (не `curl /api/tags`).** В `ollama/ollama:latest` может не быть curl/wget. `ollama` CLI — есть. `ollama list` — лёгкий, проверяет сервер.

**Почему `retries: 30`, `start_period: 30s`.** Ollama стартует 10–20 с. `start_period` — grace. 30 × 10 с = 300 с до `unhealthy`. Модели уже в volume — не ждать. 5 мин — что-то не так → unhealthy.

**Почему `ollama-init` с `depends_on: ollama: service_healthy`.** Init не должен `ollama pull`, пока ollama не готова. `service_healthy` ждёт healthcheck.

**Почему `command: >-` (folded scalar).** Многострочный shell-скрипт. `>-` схлопывает `\n` в пробелы. CRLF-ошибки Windows не ломают shell. Без `>-` (`|`) `\r` в конце строки сломал бы `then`/`fi`.

**Почему `$${LLM_MODEL}`, а не `${LLM_MODEL}`.** `$$` — экранирование: `docker compose` передаст `$${LLM_MODEL}` в shell контейнера. Shell подставит из контейнерного `environment`. `${LLM_MODEL}` (без `$$`) — `docker compose` подставил бы из `.env` при парсинге YAML. В нашем случае `LLM_MODEL` в `environment:` контейнера (из `${LLM_MODEL}` в compose). `$${}` — правильно.

**Почему `if [ -n "$${MODEL}" ]`.** `CRITIC_MODEL`/`EDITOR_MODEL` могут быть пустыми (`:-` → пустая строка). `-n ""` → false, `ollama pull ""` не запускается.

**Почему `if ollama list | grep -q "^$${MODEL}"`.** Проверка «модель уже есть». `grep -q` — тихий. `^$${MODEL}` — привязка к началу: `qwen2.5:7b` не матчит `qwen2.5:7b-instruct`. Это то же, что `_ollama_name_matches` в health.py.

**Почему `restart: "no"` для `ollama-init`.** Одноразовый. После `echo 'completed'` — exit 0, не перезапускается.

**Почему `redis` с `--appendonly yes`.** AOF-персистентность. Redis без AOF теряет данные при рестарте. Для Celery-очередей — важно. `redis_data` volume сохраняет AOF-файл.

**Почему `redis` в `app.depends_on`.** `app` использует redis через `/health → check_redis`. Без redis — 503. `service_healthy` — ждёт.

### Запуск

```powershell
cd D:\RUNG\infrastructure
docker compose up -d --build
```

**Первый запуск** — `ollama-init` качает модели (~10 ГБ для 14b + ~5 ГБ для 7b). Займёт 10–20 минут в зависимости от интернета.

**Что вы должны увидеть:**

```
[+] Running 6/6
 ✔ Container infrastructure-db-1       Healthy
 ✔ Container infrastructure-redis-1    Healthy
 ✔ Container infrastructure-qdrant-1   Healthy
 ✔ Container infrastructure-ollama-1   Healthy
 ✔ Container infrastructure-ollama-init-1  Exited
 ✔ Container infrastructure-app-1      Started
```

`ollama-init` — `Exited (0)` (одноразовый, отработал). Модели в `rung_ollama_data`.

Проверка:

```powershell
docker compose exec ollama ollama list
```

**Что вы должны увидеть:**

```
NAME              ID           SIZE      MODIFIED
qwen2.5:14b       ...          ~9 GB     2 minutes ago
qwen2.5:7b        ...          ~5 GB     2 minutes ago
```

### Коммит

```powershell
git add infrastructure/docker-compose.yml
git commit -m "feat(week-8): add ollama, ollama-init, redis services"
```

---

## 8.7. `backend/api/routes/health.py` — полная проверка 4 сервисов

**Что это:** обновить `/health` (Этап 3: только DB; Этап 7: + Qdrant). Теперь все 4 сервиса: `database`, `redis`, `qdrant`, `ollama`. Плюс `_ollama_name_matches` для точного сравнения имени модели.

**Зачем это нужно:** `/health` — readiness-проба Docker. Если redis упал, а `/health` — 200, оркестратор не узнает. Сейчас — 503 при падении любого.

Откройте `backend/api/routes/health.py` и замените содержимое на финальную версию:

```python
# backend/api/routes/health.py
import asyncio
import logging

from fastapi import APIRouter, Response, status
from sqlalchemy import text

from backend.core.config import get_settings
from backend.core.constants import KNOWN_LANGUAGES, LANGUAGES
from backend.db.session import async_engine
from backend.utils.http_client import get_ollama_client

logger = logging.getLogger(__name__)
router = APIRouter(tags=["health"])


@router.get("/health")
async def health(response: Response):
    """
    Liveness / readiness probe.

    Returns HTTP 200 when every dependency is reachable and HTTP 503
    otherwise, so orchestrators (Kubernetes, Docker healthcheck, ...) can
    act on the status code alone. The body carries per-service details.
    """
    settings = get_settings()

    db_ok, db_error = await check_database()
    redis_ok, redis_error = await check_redis(settings.REDIS_URL)
    qdrant_ok, qdrant_error = await check_qdrant(settings.qdrant_url)
    ollama_ok, ollama_error = await check_ollama(
        settings.ollama_url, settings.LLM_MODEL
    )

    all_healthy = all((db_ok, redis_ok, qdrant_ok, ollama_ok))
    response.status_code = (
        status.HTTP_200_OK if all_healthy else status.HTTP_503_SERVICE_UNAVAILABLE
    )

    return {
        "status": "healthy" if all_healthy else "unhealthy",
        "project": settings.PROJECT_NAME,
        "model": settings.LLM_MODEL,
        "ensemble_models": settings.ensemble_models_list,
        "debug": settings.DEBUG,
        "supported_languages": LANGUAGES,
        "known_languages": KNOWN_LANGUAGES,
        "services": {
            "database": {"healthy": db_ok, "error": db_error},
            "redis": {"healthy": redis_ok, "error": redis_error},
            "qdrant": {"healthy": qdrant_ok, "error": qdrant_error},
            "ollama": {"healthy": ollama_ok, "error": ollama_error},
        },
    }


async def check_database():
    try:
        async with async_engine.connect() as conn:
            await conn.execute(text("SELECT 1"))
        return True, None
    except Exception as e:
        logger.error("Database health check failed: %s", e)
        return False, str(e)


async def check_redis(redis_url: str):
    try:
        import redis.asyncio as redis

        client = redis.from_url(redis_url, socket_connect_timeout=3)
        try:
            await client.ping()
        finally:
            await client.aclose()
        return True, None
    except Exception as e:
        logger.error("Redis health check failed: %s", e)
        return False, str(e)


async def check_qdrant(qdrant_url: str):
    """
    Verify that Qdrant is reachable.

    Runs the synchronous QdrantClient in a worker thread so it does not
    block the event loop. Passes ``QDRANT_API_KEY`` when configured, so
    an authenticated Qdrant reports healthy.
    """
    settings = get_settings()

    def _check() -> None:
        from qdrant_client import QdrantClient

        client_kwargs = {"url": qdrant_url, "timeout": 3}
        if settings.QDRANT_API_KEY:
            client_kwargs["api_key"] = settings.QDRANT_API_KEY

        client = QdrantClient(**client_kwargs)
        try:
            client.get_collections()
        finally:
            try:
                client.close()
            except Exception:
                pass

    try:
        await asyncio.to_thread(_check)
        return True, None
    except Exception as e:
        logger.error("Qdrant health check failed: %s", e)
        return False, str(e)


def _ollama_name_matches(requested: str, available: str) -> bool:
    """
    Exact-match comparison for Ollama model names.

    Substring matching is wrong here: ``"qwen2.5:7b" in "qwen2.5:7b-instruct"``
    is True, so an /health probe would report healthy while the model that
    the agents actually call is not present, and translation silently
    degrades to the Hugging Face fallback. The reverse case
    (``"qwen"`` matching every model that contains "qwen") is equally
    misleading.

    Ollama resolves a bare name ``foo`` to ``foo:latest``, so both forms
    are accepted:

        requested="qwen2.5:7b"  available="qwen2.5:7b"          -> True
        requested="qwen2.5:7b"  available="qwen2.5:7b-instruct" -> False
        requested="qwen2.5"     available="qwen2.5:latest"      -> True
        requested="qwen2.5"     available="qwen2.5"             -> True
        requested="qwen"        available="qwen2.5:7b"          -> False
    """
    if requested == available:
        return True

    # ``name`` and ``name:latest`` refer to the same model in Ollama.
    if ":" not in requested and available == f"{requested}:latest":
        return True
    if ":" not in available and requested == f"{available}:latest":
        return True

    return False


async def check_ollama(ollama_url: str, model_name: str):
    """
    Verify that Ollama is reachable and that ``model_name`` is available.

    Uses the shared per-loop httpx client and overrides its timeout per
    request, so probes stay fast without creating a fresh client on every
    call. The ``ollama/`` prefix is stripped before the comparison.
    """
    normalised = model_name.replace("ollama/", "") if model_name else ""
    try:
        tags_url = ollama_url.replace("/api/generate", "/api/tags")
        client = get_ollama_client()
        response = await client.get(tags_url, timeout=5.0)
        response.raise_for_status()
        data = response.json()
        models = [m["name"] for m in data.get("models", [])]
        if not any(_ollama_name_matches(normalised, m) for m in models):
            return False, f"Model '{normalised}' not found"
        return True, None
    except Exception as e:
        logger.error("Ollama health check failed: %s", e)
        return False, str(e)
```

### Разбор ключевых решений

**Почему `_ollama_name_matches` — точное сравнение, а **не** `in`.** Самое важное:
- `"qwen2.5:7b" in "qwen2.5:7b"` → True ✓
- `"qwen2.5:7b" in "qwen2.5:7b-instruct"` → True ✗ (health врёт: агент вызывает `qwen2.5:7b`, но в Ollama только `qwen2.5:7b-instruct`, реальный вызов упадёт с `model not found`)
- `"qwen" in "qwen2.5:7b"` → True ✗

`_ollama_name_matches` явно:
1. `requested == available` → True.
2. `requested="qwen2.5"` (без `:`), `available="qwen2.5:latest"` → True (Ollama резолвит bare name).
3. Обратное: `requested="qwen2.5:latest"`, `available="qwen2.5"` → True.
4. Иначе — False.

**Почему `":" not in requested`.** `"qwen2.5"` (без `:`) — Ollama резолвит в `"qwen2.5:latest"`. `":" not in requested` — покрывает оба случая. Если в `requested` нет `:`, добавляем `:latest` для сравнения.

**Почему `normalised = model_name.replace("ollama/", "")`.** `LLM_MODEL=ollama/qwen2.5:7b` (префикс для LiteLLM). Ollama API возвращает `qwen2.5:7b` без префикса. Снимаем — совпадает.

**Почему `check_redis` через `redis.asyncio`, а не sync.** Sync-клиент блокирует event loop на `ping()` (если redis недоступен — 3 с `socket_connect_timeout`). Async — не блокирует.

**Почему `socket_connect_timeout=3`.** Redis в Docker — < 100 мс. 3 с — «недоступен».

**Почему `try/finally` с `aclose()`.** `client.ping()` может бросить (redis недоступен), но клиент нужно закрыть. Без `finally` — утечка.

**Почему `check_qdrant` через `asyncio.to_thread`.** `qdrant-client` — sync. В event loop'е — блокирует всё на время запроса. `to_thread` — выносит в threadpool.

**Почему `check_qdrant` передаёт `api_key`, если задан.** Если Qdrant с auth — запрос без ключа → 401 → health 503, хотя сервис жив. `client_kwargs["api_key"]` — правильный клиент.

**Почему `check_ollama` через `get_ollama_client()`.** Per-loop клиент с 300 с timeout. Переиспользуем, `timeout=5.0` переопределяем на конкретный запрос. Не создаём новый клиент на каждый healthcheck.

**Почему `timeout=5.0` на `client.get`.** Перебивает дефолтный 300 с. Healthcheck — быстрый (Docker проверяет каждые 15 с). 5 с — «ollama недоступна» быстрее, чем за 300 с.

**Почему `tags_url = ollama_url.replace("/api/generate", "/api/tags")`.** `ollama_url = http://host:port/api/generate`. `/api/tags` — список моделей. Простая замена.

**Почему `if not any(_ollama_name_matches(...))` → `False, "Model not found"`.** Модель отсутствует — 503. Это проверка **именно модели агента**, не просто «Ollama жива». Ollama может быть жива с `qwen2.5:7b`, но `LLM_MODEL=qwen2.5:14b` — unhealthy (правильно).

**Почему `KNOWN_LANGUAGES` в `/health`.** Клиент (Streamlit, Этап 12) читает `/health.known_languages` для filter'а «Ollama-ready» (Ensemble disabled при неподдерживаемой паре). `LANGUAGES` — все 18.

**Почему порядок проверок `db, redis, qdrant, ollama` — последовательно.** Не параллельно (`asyncio.gather`). Если db упала за 5 с, redis проверяется следующей 5 с — 20 с worst case. Параллельно было бы 5 с, но `check_database` использует `async_engine` — параллельные запросы конкурируют за пул. Простота > скорость. Docker healthcheck timeout 10 с — 20 с не влезает. **Ограничение** (для production нужен `asyncio.gather` с `return_exceptions=True`), для учебного — приемлемо.

### Проверка

```powershell
curl http://localhost:8000/health
```

**Что вы должны увидеть** (при всех сервисах):

```json
{
  "status":"healthy",
  "project":"RUNG",
  "model":"qwen2.5:14b",
  "ensemble_models":["qwen2.5:7b","mistral:7b-instruct","llama3.1:8b"],
  "debug":true,
  "supported_languages":{"en":"English",...},
  "known_languages":["en","ru","fa",...],
  "services":{
    "database":{"healthy":true,"error":null},
    "redis":{"healthy":true,"error":null},
    "qdrant":{"healthy":true,"error":null},
    "ollama":{"healthy":true,"error":null}
  }
}
```

**Эксперимент: сломайте redis.**

```powershell
docker compose stop redis
curl http://localhost:8000/health
```

**Что вы должны увидеть** (503, `redis.healthy: false`):

```json
{"status":"unhealthy",...,"services":{"redis":{"healthy":false,"error":"..."},...}}
```

**Верните:** `docker compose start redis`.

**Эксперимент: `_ollama_name_matches`.**

```powershell
python -m poetry run python -c "from backend.api.routes.health import _ollama_name_matches; print(_ollama_name_matches('qwen2.5:7b', 'qwen2.5:7b')); print(_ollama_name_matches('qwen2.5:7b', 'qwen2.5:7b-instruct')); print(_ollama_name_matches('qwen2.5', 'qwen2.5:latest')); print(_ollama_name_matches('qwen', 'qwen2.5:7b'))"
```

**Что вы должны увидеть:**

```
True
False    ← не substring
True     ← name vs name:latest
False    ← не substring
```

### Коммит

```powershell
git add backend/api/routes/health.py
git commit -m "feat(week-8): full /health with db, redis, qdrant, ollama + exact name match"
```

---

## 8.8. `backend/api/main.py` — `close_ollama_clients` в `lifespan`

**Что это:** в `lifespan.finally` (был `pass` в Этапе 3) — добавить `await close_ollama_clients()`.

**Зачем это нужно:** `get_ollama_client()` создаёт `httpx.AsyncClient` на loop. При shutdown FastAPI клиент нужно закрыть — иначе TCP-соединения висят в `CLOSE_WAIT`, через несколько рестартов — `too many open files`.

Откройте `backend/api/main.py`. Обновите:

**1. Импорт:**

```python
from backend.utils.http_client import close_ollama_clients
```

**2. В `lifespan` — заменить `finally: pass` на:**

```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    async with async_engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
        for statement in _POST_CREATE_STATEMENTS:
            await conn.execute(text(statement))
    try:
        yield
    finally:
        await close_ollama_clients()
```

### Разбор

**Почему `finally`, а не после `yield`.** Гарантирует вызов даже при исключении. Без `finally` при краше клиент не закроется — утечка.

**Почему `await close_ollama_clients()`, а не `asyncio.run(...)`.** `lifespan` — async-контекст. `close_ollama_clients` — async. `await` — прямой вызов.

**Почему только httpx, а не `async_engine.dispose()`.** `close_ollama_clients` закрывает сетевые ресурсы. `async_engine` (SQLAlchemy) — пул соединений, GC + ОС закроют при shutdown процесса. В референсе `dispose()` нет — **ограничение** (в тестах многократный restart может не очищать пул до GC, но для production — приемлемо).

---

## 8.9. Тесты

Тесты скинет пользователь. Здесь — что должно проверяться.

### `tests/unit/test_llm_client.py`

- `call_ollama_with_model` — успех (mock httpx): возвращает `response.json()["response"]`, очищенный.
- Timeout на первой попытке, успех на второй → результат второй.
- Timeout на всех 3 → raise последнего.
- `HTTPStatusError` (500) на всех 3 → raise.
- `ConnectError` → 3 попытки, `retry_delay=5.0` (mock `asyncio.sleep`).
- Пустой `response` в JSON → `""`, не raise.
- `model_name="ollama/qwen2.5:7b"` → payload `"model": "qwen2.5:7b"`.
- `seed=42` по умолчанию → `options["seed"] == 42`.
- `seed=None` → `options` без `seed`.
- `clean=False` → raw (без `clean_translation`).

**Mock:** `monkeypatch.setattr("backend.utils.llm_client.get_ollama_client", lambda: mock_client)`, `mock_client.post` — `AsyncMock`. `monkeypatch.setattr("backend.utils.llm_client.asyncio.sleep", AsyncMock())` — не ждать 5 с.

### `tests/unit/test_clean_translation.py`

- JSON `{"translation": "привет"}` → `"привет"`.
- Fenced ` ```json\n{"translation": "привет"}\n``` ` → `"привет"`.
- Префикс `Translation: привет` → `"привет"`.
- `Перевод: привет` → `"привет"`.
- `Here is the translation: привет` → `"привет"`.
- Обрамляющие кавычки `"привет"` → `"привет"`.
- Tokenizer `привет LBLBLB_YYZBB` → `"привет"`.
- CJK-символы `"привет 你好"` → `"привет"`.
- CJK-пунктуация `"привет，мир。"` → `"приветмир"`.
- Пустой результат `"，。"` → `original.strip()` = `"，。"`.
- Пустая строка `""` → `""`.

### `tests/unit/test_translator_manager.py`

- `_is_ollama_available` кэширует на 30 с (mock `time.monotonic`).
- После TTL — повторный probe.
- `get_translator_for_engine("huggingface", ...)` → `Callable` (HF).
- `get_translator_for_engine("ollama", "en", "ru")` при доступной Ollama → `None`.
- `get_translator_for_engine("ollama", "en", "udm")` (udm не в KNOWN) → `Callable` (HF, fallback).
- `get_translator_for_engine("ollama", "en", "ru")` при недоступной + `FALLBACK_TO_HF_ON_ERROR=true` → `Callable`.
- `get_translator_for_engine("ollama", "en", "ru")` при недоступной + `FALLBACK_TO_HF_ON_ERROR=false` → `raise EngineUnavailableError`.
- `get_translator_for_engine("auto", "en", "ru")` при доступной → `None`.
- `get_translator_for_engine("auto", "en", "udm")` → `Callable`.
- `get_translator_for_engine("unknown", ...)` → warning, как auto.

**Mock:** `monkeypatch.setattr(translator_manager, "_is_ollama_available", AsyncMock(return_value=True))`.

### `tests/unit/test_health_ollama.py`

- `_ollama_name_matches("qwen2.5:7b", "qwen2.5:7b")` → True.
- `("qwen2.5:7b", "qwen2.5:7b-instruct")` → False.
- `("qwen2.5", "qwen2.5:latest")` → True.
- `("qwen2.5:latest", "qwen2.5")` → True.
- `("qwen", "qwen2.5:7b")` → False.
- `("ollama/qwen2.5:7b", "qwen2.5:7b")` → True (после replace).
- `check_ollama`: модель есть → `(True, None)`; нет → `(False, "Model ... not found")`; ConnectError → `(False, "...")`.

### `conftest.py` — фикстура `autouse`

```python
@pytest.fixture(autouse=True)
def clean_hf_models():
    yield
    from backend.utils.hf_llm import unload_hf_llm
    unload_hf_llm()
```

Без неё `HuggingFaceLLM` singleton остаётся между тестами — `pytest` съест RAM.

---

## 8.10. Финальная проверка

```powershell
python -m poetry run pytest tests/ -v
```

Все тесты Этапов 1–8 зелёные.

Финальный чек-лист:

- [ ] `docker compose ps` — `ollama`, `redis`, `qdrant`, `db` в `healthy`, `ollama-init` в `Exited (0)`.
- [ ] `docker compose exec ollama ollama list` — 2 модели из `.env`.
- [ ] `curl http://localhost:8000/health` — 200, все `services.*.healthy: true`.
- [ ] `docker compose stop redis` → `/health` 503 → `start redis` → 200.
- [ ] `docker compose stop ollama` → `/health` 503 → `start ollama` → 200.
- [ ] `Critic._call_llm.__qualname__ == "Agent._call_llm"`.
- [ ] `Editor._call_llm.__qualname__ == "Agent._call_llm"`.
- [ ] `unload_hf_llm()` работает.
- [ ] `pytest` — зелёный.

**Демонстрация real LLM:**

```powershell
python -m poetry run python -c "
import asyncio
from backend.utils.llm_client import call_ollama_with_model
async def m():
    r = await call_ollama_with_model(
        'Translate from English to Russian: Hello world\nTranslation:',
        'qwen2.5:7b',
    )
    print('Translation:', r)
asyncio.run(m())
"
```

**Что вы должны увидеть** (реальный перевод):

```
Translation: Привет мир
```

Коммит и merge:

```powershell
git add -A
git commit -m "feat(week-8): complete LLM integration (Ollama + HF, real calls)"
git push -u origin week-8
```

---

## Что дальше

**Этап 9** — `perform_translation` в `backend/services/pipeline.py` с 8 ablation-режимами. Заглушки сняты, `Critic`/`Editor` реально вызывают LLM. `Supervisor` работает с реальными score. Роуты `/translate`, `/translate/multi`, `/translate/batch`, `/translate/ensemble` подключаются к API.

**Этап 10** — HITL: `ActiveLearningManager` (уже создан) + роуты `/hitl/*`.

**Этап 11** — Celery worker + beat (redis уже поднят).

**Этап 12** — Streamlit + nginx + README.